In [0]:
%run ./utilities

In [0]:
bronze_table_name = "meteo.bronze.raw_weather"

In [0]:
%sql
use schema bronze

In [0]:
fetch_weather_to_landing()

In [0]:
%sql
create schema if not exists meteo.bronze


In [0]:
weather_stream_df = read_weather_stream()

In [0]:

from pyspark.sql.functions import current_timestamp, lit, col

weather_bronze_df = (
    weather_stream_df
    .withColumn("bronze_ingestion_timestamp", current_timestamp())
    .withColumn("source_file_name", col("_metadata.file_path"))
    .withColumn("source_system", lit("open_meteo_api"))
)


In [0]:
bronze_query = (
    weather_bronze_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_bronze)
    .trigger(availableNow=True)
    .toTable(bronze_table_name)
)

bronze_query.awaitTermination()

In [0]:
%sql
select * from meteo.bronze.raw_weather